# Week 7 Exercise — Fine-tuning data & evaluation for Africa / Côte d'Ivoire

**Author:** asket  
**Location:** `community_contributions/asket/`

**Not Week 5 (RAG):** Week 5 = retrieval + chat over a knowledge base (Q&A). **Week 7 = fine-tuning** a model for **price prediction**: we prepare **prompt + completion** training data (XOF), export it for Colab/QLoRA, then **evaluate** baseline vs LLM and compare errors.

This notebook prepares **training-ready data** (Day 2) and runs **evaluation** (Day 5) for price prediction in **XOF** (Côte d'Ivoire / West Africa). Use case: **market price transparency**; data can be used to fine-tune an open-source model (e.g. Llama + QLoRA in Colab) for local, low-cost inference.

---

## Week 7 roadmap

| Day | Topic |
|-----|--------|
| Day 1 | QLoRA |
| Day 2 | Prompt data and base model |
| Day 3–4 | Train (fine-tuning) |
| Day 5 | Eval |

Here: **Day 2** (prompt/completion format, export for training) + **Day 5** (eval, model comparison). Full training runs in Colab (see main repo).

## Setup — Imports and config

**Dependencies:** openai, python-dotenv, gradio, pandas, plotly, scikit-learn (or use notebook install cell).

In [13]:
# Install (run once if needed); on Homebrew Python use --user --break-system-packages
# %pip install -q openai python-dotenv gradio pandas plotly scikit-learn --user --break-system-packages

In [14]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import mean_absolute_error, r2_score

load_dotenv(override=True)

MODEL = "gpt-4o-mini"
CURRENCY = "XOF"
PREFIX_XOF = "Prix en XOF "
QUESTION_FR = "Quel est le prix de ce produit en francs CFA (XOF) ? Réponds par un seul nombre."
QUESTION_EN = "What is the price of this product in West African CFA francs (XOF)? Reply with a single number."
DEFAULT_N_EVAL = 15

In [15]:
if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
    print("Using OpenRouter")
else:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    print("Using OpenAI")

Using OpenAI


## Data — West Africa / Côte d'Ivoire (XOF)

Load product descriptions and prices in **XOF**. Same prompt format as Week 7 (question + summary + prefix + completion) but in XOF for local relevance.

In [16]:
def _parse_row(row: str):
    idx = row.rfind(",")
    if idx == -1:
        return None
    text = row[:idx].strip().strip('"')
    try:
        price = float(row[idx + 1:].strip())
    except ValueError:
        return None
    return (text, price)

def load_data_xof():
    """Load (text, price) in XOF. Tries week6/human_out.csv or week7 path; fallback = West Africa sample."""
    for path in [Path("../../human_out.csv"), Path("week6/human_out.csv"), Path("../human_out.csv")]:
        if path.exists():
            items = []
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    p = _parse_row(line)
                    if p: items.append({"text": p[0], "price": p[1]})
            if items:
                print(f"Loaded {len(items)} items from {path} (XOF)")
                return items
    # Fallback: West Africa / Côte d'Ivoire sample (XOF)
    sample = [
        ("Title: Riz 25 kg (parboiled, local)\nCategory: Alimentation\nDescription: Sac de 25 kg de riz parboiled, marché ivoirien.", 12500),
        ("Title: Lampe solaire LED 5W\nCategory: Électronique\nDescription: Lampe rechargeable solaire pour éclairage hors réseau.", 3500),
        ("Title: Huile de cuisson 1L\nCategory: Alimentation\nDescription: Bouteille d'huile végétale 1 litre.", 1200),
        ("Title: Ciment 50 kg\nCategory: BTP\nDescription: Sac de ciment Portland 50 kg.", 4500),
        ("Title: Téléphone Android d'occasion 4G\nCategory: Électronique\nDescription: Smartphone Android reconditionné, 2 Go RAM.", 35000),
        ("Title: Cartable écolier\nCategory: Accessoires\nDescription: Cartable solide pour écoliers.", 5500),
        ("Title: Moustiquaire imprégnée\nCategory: Santé\nDescription: Moustiquaire imprégnée 1 ou 2 places.", 2500),
        ("Title: Casque moto\nCategory: Automobile\nDescription: Casque de sécurité moto.", 8500),
        ("Title: Farine de maïs 1 kg\nCategory: Alimentation\nDescription: Paquet 1 kg pour bouillie.", 600),
        ("Title: Chaises plastique lot de 4\nCategory: Mobilier\nDescription: 4 chaises plastique pour intérieur/extérieur.", 18000),
        ("Title: Cacao 1 kg fèves\nCategory: Agriculture\nDescription: Fèves de cacao Côte d'Ivoire 1 kg.", 2200),
        ("Title: Savon de lessive 1 kg\nCategory: Hygiène\nDescription: Savon en barre pour lessive.", 800),
        ("Title: Pile AA lot 4\nCategory: Électronique\nDescription: 4 piles AA alcalines.", 1500),
        ("Title: Bidon eau 20L\nCategory: Alimentation\nDescription: Eau potable 20 litres.", 500),
        ("Title: Télévision LED 32 pouces\nCategory: Électronique\nDescription: TV LED 32 pouces entrée de gamme.", 95000),
    ]
    items = [{"text": t, "price": p} for t, p in sample]
    print(f"Using West Africa / Côte d'Ivoire sample: {len(items)} items (XOF)")
    return items

data = load_data_xof()

Using West Africa / Côte d'Ivoire sample: 15 items (XOF)


## Prompt format (Day 2 style)

Week 7 uses: **Question** + **summary** + **Prefix** + **completion**. For Africa we use the same structure in XOF (French or English).

In [17]:
def make_prompt_completion(item: dict, lang: str = "fr") -> tuple:
    """Return (prompt, completion) for this item. prompt = question + text + prefix; completion = price in XOF."""
    q = QUESTION_FR if lang == "fr" else QUESTION_EN
    prompt = f"{q}\n\n{item['text']}\n\n{PREFIX_XOF}"
    completion = f"{int(round(item['price']))}"
    return prompt, completion

# Example
p, c = make_prompt_completion(data[0], "fr")
print("Prompt (extract):", p[:200], "...")
print("Completion:", c)

# Day 2 style: prompt length stats (for fine-tuning chunk/context decisions)
prompts, completions = zip(*[make_prompt_completion(d, "fr") for d in data])
len_p = [len(pr) for pr in prompts]
len_c = [len(co) for co in completions]
print(f"\nPrompt length: min={min(len_p)}, max={max(len_p)}, avg={sum(len_p)/len(len_p):.0f} chars")
print(f"Completion length: min={min(len_c)}, max={max(len_c)} chars")

Prompt (extract): Quel est le prix de ce produit en francs CFA (XOF) ? Réponds par un seul nombre.

Title: Riz 25 kg (parboiled, local)
Category: Alimentation
Description: Sac de 25 kg de riz parboiled, marché ivoirien ...
Completion: 12500

Prompt length: min=172, max=215, avg=191 chars
Completion length: min=3, max=5 chars


## Training data export (for Colab / QLoRA)

Build prompt/completion pairs and export as JSONL for fine-tuning (Week 7 Day 3–4). Week 5 (RAG) does not export training data.

In [18]:
import json

train_data = [{"prompt": make_prompt_completion(d, "fr")[0], "completion": make_prompt_completion(d, "fr")[1]} for d in data]
out_path = Path("pricer_train_xof.jsonl")
with open(out_path, "w", encoding="utf-8") as f:
    for row in train_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Exported {len(train_data)} pairs to {out_path}")
display(pd.DataFrame(train_data).head(2))

Exported 15 pairs to pricer_train_xof.jsonl


,prompt,completion
0,Quel est le prix de ce produit en francs CFA (...,12500
1,Quel est le prix de ce produit en francs CFA (...,3500


## Predictor — LLM API (stand-in before fine-tuning)

Single call per item; parse numeric answer. Replace with a **fine-tuned** model (e.g. Llama + QLoRA in Colab) for lower cost and better XOF accuracy.

In [19]:
def parse_price(reply: str) -> float:
    if not reply:
        return 0.0
    s = str(reply).replace(",", "").replace(" ", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

def predict_xof(text: str, lang: str = "fr") -> float:
    q = QUESTION_FR if lang == "fr" else QUESTION_EN
    prompt = f"{q}\n\n{text}\n\n{PREFIX_XOF}"
    try:
        r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
        return parse_price(r.choices[0].message.content)
    except Exception as e:
        print(f"API error: {e}")
        return 0.0

## Baselines and evaluation (Day 5 style)

Constant (mean) baseline; compare MAE and R² in XOF.

In [20]:
mean_price = float(np.mean([d["price"] for d in data]))

def baseline_constant(_text):
    return mean_price

def evaluate(data, predictor, n=DEFAULT_N_EVAL):
    subset = data[:n]
    truths = [d["price"] for d in subset]
    preds = [predictor(d["text"]) for d in subset]
    mae = mean_absolute_error(truths, preds)
    r2 = r2_score(truths, preds)
    return mae, r2

mae_base, r2_base = evaluate(data, baseline_constant)
print(f"Constant baseline — MAE: {mae_base:,.0f} XOF  R²: {r2_base:.3f}")

Constant baseline — MAE: 14,619 XOF  R²: 0.000


In [21]:
subset_eval = data[:DEFAULT_N_EVAL]
truths_eval = [d["price"] for d in subset_eval]
preds_llm = [predict_xof(d["text"]) for d in subset_eval]
mae_llm = mean_absolute_error(truths_eval, preds_llm)
r2_llm = r2_score(truths_eval, preds_llm)
print(f"LLM (API) — MAE: {mae_llm:,.0f} XOF  R²: {r2_llm:.3f}")

# Per-item table (reuses preds_llm above; no extra API calls)
n_show = min(8, len(subset_eval))
preds_base_sub = [baseline_constant(d["text"]) for d in subset_eval[:n_show]]
rows = []
for i in range(n_show):
    d = subset_eval[i]
    title = (d["text"].split(chr(10))[0])[:45] + ("." if len(d["text"]) > 45 else "")
    rows.append({"Product": title, "Truth": int(d["price"]), "Baseline": int(preds_base_sub[i]), "LLM": int(preds_llm[i]), "|Err| base": int(round(abs(preds_base_sub[i] - d["price"]))), "|Err| LLM": int(round(abs(preds_llm[i] - d["price"])))})
eval_df = pd.DataFrame(rows)
display(eval_df)

LLM (API) — MAE: 11,413 XOF  R²: 0.347


,Product,Truth,Baseline,LLM,|Err| base,|Err| LLM
0,"Title: Riz 25 kg (parboiled, local).",12500,12786,25000,287,12500
1,Title: Lampe solaire LED 5W.,3500,12786,15000,9287,11500
2,Title: Huile de cuisson 1L.,1200,12786,5000,11587,3800
3,Title: Ciment 50 kg.,4500,12786,5000,8287,500
4,Title: Téléphone Android d'occasion 4G.,35000,12786,50000,22213,15000
5,Title: Cartable écolier.,5500,12786,5000,7287,500
6,Title: Moustiquaire imprégnée.,2500,12786,15000,10287,12500
7,Title: Casque moto.,8500,12786,50000,4287,41500


## Results — Model comparison (Week 7 style)

Bar chart of mean absolute error (XOF) for baseline vs LLM, aligned with `results.ipynb`.

In [22]:
results = [
    ("Constant (mean)", "gray", mae_base),
    ("LLM (gpt-4o-mini)", "slateblue", mae_llm),
]
labels, colors, values = zip(*results)
fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="Erreur de prédiction (XOF) — Afrique / Côte d'Ivoire",
    yaxis=dict(title="MAE (XOF)", range=[0, max(values) * 1.15]),
    xaxis=dict(tickangle=-25),
    width=600,
    height=400,
    template="plotly_white",
)
try:
    fig.show()
except ValueError:
    fig.write_html("week7_mae_chart.html")
    print("Chart saved to week7_mae_chart.html (open in browser)")

Chart saved to week7_mae_chart.html (open in browser)


## Gradio — Compare baseline vs LLM (Week 7 eval)

Pick a sample product or enter a custom description; see **baseline (constant mean)** vs **LLM** prediction. Unlike Week 5 (chat Q&A), this is a **model comparison** demo for price prediction.

In [23]:
import gradio as gr

choices = ["Custom"] + [(d["text"].split(chr(10))[0][:50] + "…") for d in data]

def compare_models(choice: str, custom_text: str) -> str:
    if choice == "Custom":
        text = (custom_text or "").strip()
        if not text:
            return "Enter a product description or pick a sample."
        truth_str = ""
    else:
        idx = choices.index(choice) - 1
        text = data[idx]["text"]
        truth_str = f"Ground truth: {int(data[idx]['price']):,} XOF"
    base = baseline_constant(text)
    llm = predict_xof(text)
    return f"Baseline (mean): {int(base):,} XOF\nLLM: {int(llm):,} XOF\n{truth_str}"

with gr.Blocks(title="Baseline vs LLM — XOF") as demo:
    gr.Markdown("**Compare baseline vs LLM** for price prediction (XOF). Choose a sample or enter custom description.")
    sel = gr.Dropdown(choices=choices, value=choices[1], label="Sample product")
    custom = gr.Textbox(label="Custom description (used when 'Custom' is selected)", placeholder="e.g. Riz 25 kg, sac parboiled...", lines=3)
    out = gr.Textbox(label="Predictions (XOF)", lines=4)
    sel.change(lambda c: compare_models(c, ""), inputs=[sel], outputs=[out])
    custom.change(lambda c, t: compare_models(c, t), inputs=[sel, custom], outputs=[out])
    demo.load(lambda: compare_models(choices[1], ""), outputs=[out])
demo.launch()

/Users/franckasket/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Gradio — Price Estimation Pipeline Analytics

Dashboard with **Price Distribution** (histogram), **Market Segments** (pie by price tier in XOF), and **Model Scaling** (learning curve: MAE vs number of evaluation samples). Run the cell below and click **Generate/Refresh Visuals** to update charts.

In [24]:
# Price tiers for XOF (Côte d'Ivoire / West Africa)
def _tier(p):
    if p < 5_000: return "Budget (<5k XOF)"
    if p < 20_000: return "Mid-Range (5k–20k)"
    if p < 50_000: return "Premium (20k–50k)"
    return "Luxury (>50k XOF)"

def build_analytics_plots():
    prices = np.array([d["price"] for d in data])
    n = len(subset_eval)
    # 1) Price distribution histogram
    fig_dist = go.Figure()
    fig_dist.add_trace(go.Histogram(x=prices, nbinsx=min(25, max(10, len(prices)//2)), marker_color="teal", name="Count"))
    fig_dist.update_layout(
        title="Price Distribution in Dataset (XOF)",
        xaxis_title="Price (XOF)",
        yaxis_title="Count",
        template="plotly_white",
        height=400,
        showlegend=False,
    )
    # 2) Market segments pie
    tiers = [_tier(p) for p in prices]
    tier_counts = pd.Series(tiers).value_counts()
    colors = {"Budget (<5k XOF)": "#e74c3c", "Mid-Range (5k–20k)": "#f39c12", "Premium (20k–50k)": "#2ecc71", "Luxury (>50k XOF)": "#3498db"}
    fig_pie = go.Figure(go.Pie(
        labels=tier_counts.index,
        values=tier_counts.values,
        hole=0.4,
        marker_colors=[colors.get(l, "gray") for l in tier_counts.index],
        textinfo="label+percent",
    ))
    fig_pie.update_layout(
        title="Dataset Composition by Price Tier (XOF)",
        template="plotly_white",
        height=400,
    )
    # 3) Learning curve: MAE vs number of eval samples (using existing preds_llm)
    ns = np.arange(1, n + 1, dtype=int)
    mae_at_n = [mean_absolute_error(truths_eval[:k], preds_llm[:k]) for k in ns]
    fig_curve = go.Figure()
    fig_curve.add_trace(go.Scatter(x=ns, y=mae_at_n, mode="lines", name="LLM MAE", line=dict(color="royalblue")))
    fig_curve.add_hline(y=mae_base, line_dash="dash", line_color="red", annotation_text="Baseline (mean)")
    fig_curve.update_layout(
        title="Learning Curve: MAE vs Evaluation Samples (XOF)",
        xaxis_title="Number of Evaluation Examples",
        yaxis_title="Mean Absolute Error (XOF)",
        template="plotly_white",
        height=400,
    )
    return fig_dist, fig_pie, fig_curve

import gradio as gr

with gr.Blocks(title="Price Estimation Pipeline Analytics") as demo_analytics:
    gr.Markdown("# Price Estimation Pipeline Analytics")
    gr.Markdown(f"**Baseline MAE:** {mae_base:,.0f} XOF   **Average Item Price:** {mean_price:,.0f} XOF")
    with gr.Tabs():
        with gr.Tab("Price Distribution"):
            plot_dist = gr.Plot(label="Price Distribution")
        with gr.Tab("Market Segments"):
            plot_pie = gr.Plot(label="Market Segments")
        with gr.Tab("Model Scaling"):
            plot_curve = gr.Plot(label="Model Scaling")
    btn = gr.Button("Generate/Refresh Visuals")

    def refresh():
        return build_analytics_plots()

    btn.click(fn=refresh, outputs=[plot_dist, plot_pie, plot_curve])
    demo_analytics.load(fn=refresh, outputs=[plot_dist, plot_pie, plot_curve])

demo_analytics.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
